In [ ]:
# Exploratory Data Analysis for Phishing Detection Dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from pathlib import Path
import json

#Setting the style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Resolve paths so the notebook works from either the project root or the notebooks folder.
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

FIG_DIR = ROOT / "data" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR = ROOT / "data" / "processed"
DATA_PATH = PROC_DIR / "phishing_corpus.csv"
STATS_PATH = PROC_DIR / "dataset_stats.json"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Processed dataset not found at {DATA_PATH}. "
        "Run scripts/prepare_dataset.py first to generate data/processed/phishing_corpus.csv."
    )
if not STATS_PATH.exists():
    raise FileNotFoundError(
        f"Dataset statistics not found at {STATS_PATH}. "
        "Run scripts/prepare_dataset.py first to generate data/processed/dataset_stats.json."
    )

print("="*60)
print("PHISHING EMAIL DETECTION - EXPLORATORY DATA ANALYSIS")
print("="*60)

# ===== LOAD DATA =====
print("
[1] Loading processed dataset...")
df = pd.read_csv(DATA_PATH)

# Keep the notebook compatible with raw/older exports if the text column name changes.
if "text" not in df.columns and "text_combined" in df.columns:
    df["text"] = df["text_combined"]

required_columns = {"text", "label"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

# Load the statistics generated by the preprocessing script.
with open(STATS_PATH, "r", encoding="utf-8") as f:
    stats = json.load(f)

print(f"Loaded {len(df):,} emails from {DATA_PATH}")
print(f"  - Legitimate: {stats['class_distribution']['legitimate_count']:,}")
print(f"  - Phishing: {stats['class_distribution']['phishing_count']:,}")

# Create label names
label_map = {0: "Legitimate", 1: "Phishing"}
df["label_name"] = df["label"].map(label_map)

# 1. CLASS DISTRIBUTION 
print("\n[2] Analyzing class distribution...")

fig, ax = plt.subplots(figsize=(8, 5))
counts = df['label_name'].value_counts()
colors = ['#2ecc71', '#e74c3c']  # Green for legitimate, Red for phishing
bars = ax.bar(counts.index, counts.values, color=colors, alpha=0.8, edgecolor='black')

# Value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}\n({height/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontweight='bold')

ax.set_ylabel("Number of Emails", fontsize=12, fontweight='bold')
ax.set_title("Class Distribution", fontsize=14, fontweight='bold', pad=20)
ax.set_ylim(0, max(counts.values) * 1.15)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_class_distribution.png", dpi=300, bbox_inches='tight')
plt.show()

# 2. EMAIL LENGTH DISTRIBUTION 
print("\n[3] Analyzing email length distribution...")

df["char_length"] = df["text"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist([df[df['label']==0]['char_length'], df[df['label']==1]['char_length']], 
             bins=50, label=['Legitimate', 'Phishing'], color=['#2ecc71', '#e74c3c'], 
             alpha=0.7, edgecolor='black')
axes[0].set_xlabel("Email Length (characters)", fontweight='bold')
axes[0].set_ylabel("Frequency", fontweight='bold')
axes[0].set_title("Email Length Distribution", fontweight='bold')
axes[0].legend()
axes[0].set_xlim(0, 5000)

# Box plot
box_data = [df[df['label']==0]['char_length'], df[df['label']==1]['char_length']]
bp = axes[1].boxplot(box_data, labels=['Legitimate', 'Phishing'], 
                      patch_artist=True, showmeans=True)
bp['boxes'][0].set_facecolor('#2ecc71')
bp['boxes'][1].set_facecolor('#e74c3c')
axes[1].set_ylabel("Email Length (characters)", fontweight='bold')
axes[1].set_title("Email Length Comparison", fontweight='bold')
axes[1].set_ylim(0, 5000)

plt.tight_layout()
plt.savefig(FIG_DIR / "02_length_distribution.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"  Legitimate - Mean: {df[df['label']==0]['char_length'].mean():.0f}, Median: {df[df['label']==0]['char_length'].median():.0f}")
print(f"  Phishing   - Mean: {df[df['label']==1]['char_length'].mean():.0f}, Median: {df[df['label']==1]['char_length'].median():.0f}")

#  3. TOP WORDS BY CLASS 
print("\n[4] Extracting top words per class...")

# Filter out noise words that appear everywhere
STOPWORDS = {
    'the', 'and', 'to', 'of', 'a', 'in', 'is', 'it', 'you', 'that', 'for', 
    'on', 'with', 'as', 'are', 'was', 'be', 'have', 'this', 'from', 'or',
    'by', 'at', 'an', 'will', 'can', 'if', 'your', 'has', 'not', 'but',
    'all', 'we', 'when', 'there', 'been', 'their', 'more', 'about', 'than'
}

def extract_top_words(texts, n=20):
    """Extract top n words, excluding stopwords"""
    word_counts = Counter()
    for text in texts:
        # Extract words with 2+ letters
        words = re.findall(r'\b[a-zA-Z]{2,}\b', str(text).lower())

        # Filter stopwords and short words
        words = [w for w in words if w not in STOPWORDS and len(w) > 2]
        word_counts.update(words)
    return word_counts.most_common(n)

top_phishing = extract_top_words(df[df["label"]==1]["text"], n=20)
top_legitimate = extract_top_words(df[df["label"]==0]["text"], n=20)

# Display the plots side by side
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Phishing words
words_p, counts_p = zip(*top_phishing)
axes[0].barh(range(len(words_p)), counts_p, color='#e74c3c', alpha=0.8, edgecolor='black')
axes[0].set_yticks(range(len(words_p)))
axes[0].set_yticklabels(words_p)
axes[0].invert_yaxis()
axes[0].set_xlabel("Frequency", fontweight='bold')
axes[0].set_title("Top 20 Words - Phishing Emails", fontweight='bold', fontsize=13)

# Legitimate words
words_l, counts_l = zip(*top_legitimate)
axes[1].barh(range(len(words_l)), counts_l, color='#2ecc71', alpha=0.8, edgecolor='black')
axes[1].set_yticks(range(len(words_l)))
axes[1].set_yticklabels(words_l)
axes[1].invert_yaxis()
axes[1].set_xlabel("Frequency", fontweight='bold')
axes[1].set_title("Top 20 Words - Legitimate Emails", fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig(FIG_DIR / "03_top_words_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

# 4. ENGINEERED FEATURES ANALYSIS
print("\n[5] Analyzing engineered features...")

features = ['url_count', 'urgency_score', 'suspicious_patterns', 'special_char_count']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, feature in enumerate(features):
    ax = axes[idx]
    
    # Box plot
    data = [df[df['label']==0][feature], df[df['label']==1][feature]]
    bp = ax.boxplot(data, labels=['Legitimate', 'Phishing'], 
                    patch_artist=True, showmeans=True)
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')
    
    # Title and labels
    title = feature.replace('_', ' ').title()
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_ylabel("Count", fontweight='bold')
    
    # Stats
    mean_legit = df[df['label']==0][feature].mean()
    mean_phish = df[df['label']==1][feature].mean()
    ax.text(0.02, 0.98, f"Legit μ={mean_legit:.2f}\nPhish μ={mean_phish:.2f}", 
            transform=ax.transAxes, va='top', bbox=dict(boxstyle='round', 
            facecolor='wheat', alpha=0.5), fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / "04_feature_analysis.png", dpi=300, bbox_inches='tight')
plt.show()

# Feature statistics table
print("\n  Feature Statistics:")
print("  " + "-"*70)
print(f"  {'Feature':<25} {'Legitimate':<20} {'Phishing':<20}")
print("  " + "-"*70)
for feature in features:
    mean_legit = df[df['label']==0][feature].mean()
    mean_phish = df[df['label']==1][feature].mean()
    print(f"  {feature:<25} {mean_legit:>8.2f} {' '*11} {mean_phish:>8.2f}")
print("  " + "-"*70)

# 5. URL PRESENCE 
print("\n[6] Analyzing URL presence...")

df["has_url"] = df["url_count"] > 0
url_presence = df.groupby("label_name")["has_url"].agg(['sum', 'count'])
url_presence['percentage'] = (url_presence['sum'] / url_presence['count']) * 100

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(url_presence.index, url_presence['percentage'], 
              color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel("Percentage of Emails with URLs (%)", fontweight='bold')
ax.set_title("URL Presence by Class", fontweight='bold', fontsize=14, pad=20)
ax.set_ylim(0, max(url_presence['percentage']) * 1.15)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_url_presence.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"  Legitimate: {url_presence.loc['Legitimate', 'percentage']:.1f}% contain URLs")
print(f"  Phishing:   {url_presence.loc['Phishing', 'percentage']:.1f}% contain URLs")

# 6. CORRELATION HEATMAP 
print("\n[7] Creating feature correlation heatmap...")

correlation_features = ['label', 'url_count', 'urgency_score', 
                        'suspicious_patterns', 'special_char_count', 'char_length']
corr_matrix = df[correlation_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Matrix", fontweight='bold', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(FIG_DIR / "06_correlation_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

#  SUMMARY FOR FINAL REPORT 
print("\n" + "="*60)
print("EDA COMPLETE - SUMMARY")
print("="*60)
print(f"\n Dataset Overview:")
print(f"  • Total emails: {len(df):,}")
print(f"  • Legitimate: {stats['class_distribution']['legitimate_count']:,} ({stats['class_distribution']['legitimate_ratio']*100:.1f}%)")
print(f"  • Phishing: {stats['class_distribution']['phishing_count']:,} ({stats['class_distribution']['phishing_ratio']*100:.1f}%)")

print(f"\n Key Findings:")
print(f"  • Phishing emails have {stats['feature_statistics']['avg_urls_phishing']:.1f}x more URLs on average")
print(f"  • Urgency keywords are {stats['feature_statistics']['avg_urgency_phishing']/max(stats['feature_statistics']['avg_urgency_legitimate'], 0.1):.1f}x more common in phishing")
print(f"  • Suspicious patterns appear {stats['feature_statistics']['avg_suspicious_phishing']/max(stats['feature_statistics']['avg_suspicious_legitimate'], 0.1):.1f}x more in phishing")

print(f"\n Visualizations saved to: {FIG_DIR}/")
print("  • 01_class_distribution.png")
print("  • 02_length_distribution.png")
print("  • 03_top_words_comparison.png")
print("  • 04_feature_analysis.png")
print("  • 05_url_presence.png")
print("  • 06_correlation_heatmap.png")

print("\n EDA complete! Ready for model training.")